# Survey Lottery Automation

A privacy-safe reconstruction of a real workflow used to process more than 1,000 survey responses. The original workflow applied a fixed serial-number rule; this public version demonstrates a reproducible seeded draw.

## 1. Safe setup

The notebook uses synthetic participant IDs only. It never connects to Google Sheets or an email server.

In [ ]:
import sys
import subprocess
from pathlib import Path

project_root = Path.cwd()
if not (project_root / 'src' / 'survey_lottery').exists():
    repository_url = 'https://github.com/Hunter20041004/survey-lottery-automation.git'
    project_root = Path('/content/survey-lottery-automation')
    if not project_root.exists():
        subprocess.run(['git', 'clone', '--depth', '1', repository_url, str(project_root)], check=True)
sys.path.insert(0, str(project_root / 'src'))

In [ ]:
from datetime import datetime, timezone
from survey_lottery.audit import build_audit_record
from survey_lottery.csv_io import load_participants
from survey_lottery.draw import run_draw
from survey_lottery.preview import build_notification_previews

## 2. Validate and draw

A caller-supplied seed makes the result reproducible and auditable. Duplicate IDs, malformed rows, and invalid draw sizes are rejected before selection.

In [ ]:
participants = load_participants(project_root / 'data' / 'sample_responses.csv')
result = run_draw(participants, winner_count=2, seed=2026)
audit = build_audit_record(
    result,
    participants,
    datetime(2026, 7, 12, tzinfo=timezone.utc),
)
audit

In [ ]:
build_notification_previews(result.winners)

## 3. Private deployment boundary

A private deployment can read Google Sheets data and map selected IDs to contact details outside this repository. Credentials belong in Colab Secrets or environment variables; real participant data and outbound email are intentionally excluded from the public demo.